# Generalized simulator — demonstration, validation & transition experiments

ONE `Simulator` covers every experiment:

| experiment | what changes |
|---|---|
| statics (k2 reproduction) | `SAVE_MAPS_ONLY`, three phases |
| dynamics (Koopman/DMD) | `SAVE_DYNAMICS` (positions + velocities) |
| quench / transition | `initial_coords` + `phases=('production',)` + new `Model` |
| any kernel / potential | swap the `Model` — simulator untouched |

**Validation targets**
1. Regression — statics path still reproduces k2 (saddle ~2.07–2.09)
2. Pilot — is the system overdamped? (decides whether gEDMD is viable)
3. Quench mechanism — in-place velocity-continuous swap, or restart fallback?
4. Transition experiments — four scenarios with quantitative order parameters

## 0. Setup

In [ ]:
from google.colab import drive
!git clone -q https://github.com/darinddv/chromatin_potential.git
!pip install -q "openmm[cuda12]" OpenMiChroM cooler cooltools
import sys; sys.path.insert(0, '/content/chromatin_potential/src')
drive.mount('/content/drive')

import os, json, numpy as np
import matplotlib.pyplot as plt
from scipy.spatial.distance import pdist, squareform
from scipy.optimize import curve_fit

from chromatin_potential.model import Model, _A_FULL, _L_FULL
from chromatin_potential.simulator import (
    Simulator, BackgroundStack, SaveSpec,
    SAVE_MAPS_ONLY, SAVE_DYNAMICS, SAVE_EVERYTHING,
    contact_probability, observed_over_expected, saddle_strength,
    radius_of_gyration, verify_inplace_quench, swap_coupling_in_context,
    MU, RC)
print('setup OK')

In [ ]:
DATA = '/content/drive/MyDrive/uky_cheng/tecsas_ablation'
OUT  = f'{DATA}/simulator_demo'
os.makedirs(OUT, exist_ok=True)
SEQ_HG38 = f'{DATA}/chr10_beads_hg38.txt'
assert os.path.exists(SEQ_HG38), SEQ_HG38

labels = np.array([l.split()[1] for l in open(SEQ_HG38) if l.strip()])
isA_chr = np.isin(labels, ['A1','A2']); isB_chr = np.isin(labels, ['B1','B2','B3','B4'])
print(f'{len(labels)} beads  A={isA_chr.sum()} B={isB_chr.sum()}')

## 1. Build the k2 model (Part A)

The `Model` is the ONLY thing that sets the potential. Everything below reuses
it; the simulator never knows which kernel produced the matrix.

In [ ]:
m5 = Model.from_type_matrix(_A_FULL, _L_FULL, real_types=[0,1,2,3,4], k=2)
A_k2 = m5.coupling_matrix()

full = _A_FULL.copy()
full[np.ix_([0,1,2,3,4],[0,1,2,3,4])] = A_k2
full[5,:5] = A_k2[4,:]; full[:5,5] = A_k2[:,4]; full[5,5] = A_k2[4,4]

ff_k2 = f'{OUT}/k2.ff'
with open(ff_k2,'w') as f:
    f.write(','.join(_L_FULL)+'\n')
    for a in range(7):
        f.write(','.join(f'{full[a,b]:.6E}' for b in range(7))+'\n')
print(open(ff_k2).read())

## 2. Regression: the statics path is unchanged

`SAVE_MAPS_ONLY` + three phases = the protocol that gave 2.073.
Two replicas is enough to catch a refactor regression (noisier than 20).

In [ ]:
sim = Simulator(seq_file=SEQ_HG38, out_dir=OUT, platform='cuda',
                background=BackgroundStack())
for s in range(2):
    sim.run_replica('k2stat', s, types_table=ff_k2, save_spec=SAVE_MAPS_ONLY)

P = sim.pool('k2stat')
print(f"\nstatics saddle (2 reps) = "
      f"{saddle_strength(observed_over_expected(P), isA_chr, isB_chr):.3f}"
      f"   target ~2.07-2.09")

### 2a. Run metadata is recorded on every run

In [ ]:
meta = sim.load_metadata('k2stat', 0)
for k in ['compute_platform','integrator','timestep','friction_gamma','temperature',
          'particle_mass_0','n_particles','cm_motion_remover','openmm_version',
          'openmichrom_version','convergence','wall_minutes']:
    print(f'{k:22s}: {meta.get(k)}')
print('\nSystem forces:', meta.get('forces'))

## 3. PILOT — the overdamped check

**The question that could invalidate gEDMD**, answered by one short run.
Saves positions + velocities + forces every step and tests whether v ≈ F/(γm).

In [ ]:
pilot_spec = SaveSpec(contact_map=False, trajectory=True, velocities=True,
                      forces=True, interval=1, monitor=True)
sim.run_replica('pilot', 0, types_table=ff_k2,
                phases=('collapse','equil','production'),
                n_production=2000, save_spec=pilot_spec, overwrite=True)

d, md_ = sim.load_trajectory('pilot', 0), sim.load_metadata('pilot', 0)
x, v, f = d['xyz'], d['vel'], d['frc']
print('shapes  xyz', x.shape, ' vel', v.shape, ' frc', f.shape)

In [ ]:
_num = lambda s: float(str(s).split()[0])
gamma, mass = _num(md_['friction_gamma']), _num(md_['particle_mass_0'])
pred = f / (gamma * mass)
r = np.corrcoef(pred.ravel(), v.ravel())[0,1]
ratio = np.linalg.norm(pred)/np.linalg.norm(v)
print(f'gamma={gamma}  mass={mass}')
print(f'corr(v, F/(gamma*m)) = {r:.4f}')
print(f'magnitude ratio      = {ratio:.4f}')
print(f'|F| mean/max = {np.abs(f).mean():.4g}/{np.abs(f).max():.4g}  finite={np.isfinite(f).all()}')
print(f'positions {x.dtype}  finite={np.isfinite(x).all()}')
print('\n--- interpretation ---')
if r > 0.9 and 0.5 < ratio < 2.0:
    print('STRONGLY OVERDAMPED: gEDMD on positions alone is well founded.')
elif r > 0.6:
    print('PARTIALLY OVERDAMPED: usable; check the underdamped formulation.')
else:
    print('NOT OVERDAMPED: inertia matters. Use the underdamped generator or')
    print('include velocities in the state vector.')

In [ ]:
fig, ax = plt.subplots(1,3, figsize=(13,3.4))
ax[0].plot(md_['rg_trace']); ax[0].set_title('Rg trace (monitor)'); ax[0].set_xlabel('frame')
ax[1].hist(np.abs(f).ravel(), bins=60); ax[1].set_yscale('log'); ax[1].set_title('|force|')
idx = np.random.default_rng(0).choice(v.size, 4000, replace=False)
ax[2].scatter(pred.ravel()[idx], v.ravel()[idx], s=2, alpha=.3)
lim = np.percentile(np.abs(v), 99.5); ax[2].plot([-lim,lim],[-lim,lim],'r--',lw=1)
ax[2].set_xlabel('F/(gamma m)'); ax[2].set_ylabel('v'); ax[2].set_title(f'overdamped r={r:.3f}')
plt.tight_layout(); plt.show()

## 4. Dynamics-mode run (chr10)

Same simulator, different `SaveSpec`. float32 positions (never float16 — these get
finite-differenced) plus velocities (stochastic, unrecoverable later).

In [ ]:
dyn_spec = SaveSpec(contact_map=True, trajectory=True, velocities=True,
                    forces=False, interval=100, monitor=True)
sim.run_replica('dyn', 0, types_table=ff_k2, n_production=100_000,
                save_spec=dyn_spec, overwrite=True)
d = sim.load_trajectory('dyn', 0)
print('arrays:', list(d.keys()), '| xyz', d['xyz'].shape, d['xyz'].dtype)

In [ ]:
xyz = d['xyz']; vv = d['vel']
lags = np.unique(np.geomspace(1, len(xyz)//3, 12).astype(int))
msd = [np.mean(np.sum((xyz[l:]-xyz[:-l])**2, axis=-1)) for l in lags]
vac = np.array([np.mean(np.sum(vv[t:]*vv[:len(vv)-t], axis=-1)) for t in range(30)])
vac /= vac[0]

fig, ax = plt.subplots(1,2, figsize=(9,3.4))
ax[0].loglog(lags, msd, 'o-'); sl = np.polyfit(np.log(lags), np.log(msd),1)[0]
ax[0].set_title(f'MSD slope={sl:.2f} (paper: 0.29)'); ax[0].set_xlabel('lag (frames)')
ax[1].plot(vac,'o-'); ax[1].axhline(0,c='k',lw=.5); ax[1].set_title('velocity autocorrelation')
plt.tight_layout(); plt.show()
print('One short replica — indicative only.')

---
# 5. Transition experiments

A 500-bead model system with **four quench scenarios**, designed so each isolates
a different aspect of the transition:

| scenario | equilibrate under | quench to | question |
|---|---|---|---|
| **control** | types A | types A (identical) | null — do order parameters drift on their own? |
| **identity swap** | `A1 A2 B1 B2` | `B1 B2 A1 A2` | which half is compact flips (compaction swap) |
| **rearrangement** | `A1 A2 B1 B2` | `A1 B1 A2 B2` | *who contacts whom* changes (contact topology) |
| **mitotic exit** | constant coupling (no compartments) | types A | **de novo** compartment formation |

The **control is not a formality** — it is the null that makes the other three
interpretable. Without it you cannot distinguish a transition from ordinary
equilibrium fluctuation.

All four differ only in **C** (per-bead coordinates), never in Λ or the physics —
the θ = (C, Λ, kernel) keystone. Because every coupling matrix is 500×500 with a
shared sequence file, the shape is fixed across the swap, which is also what makes
the in-place velocity-continuous quench possible.

### 5.0 Build the four models

In [ ]:
N_BEADS = 500; blk = N_BEADS // 4
cfg = {
 'A'      : ['A1']*blk + ['A2']*blk + ['B1']*blk + ['B2']*blk,   # A-half / B-half
 'Bswap'  : ['B1']*blk + ['B2']*blk + ['A1']*blk + ['A2']*blk,   # identities swapped
 'Cinter' : ['A1']*blk + ['B1']*blk + ['A2']*blk + ['B2']*blk,   # interleaved
}

names  = [f't{i:05d}' for i in range(N_BEADS)]
SEQ500 = f'{OUT}/perbead_500.txt'
with open(SEQ500,'w') as fh:
    for i,n in enumerate(names): fh.write(f'{i+1} {n}\n')

models = {k: m5.expand_to_beads(v) for k,v in cfg.items()}

# 'flat' = constant coupling: SAME mean attraction, ZERO differential.
# This is the compartment-free (mitotic-like) state. Using the mean keeps
# overall compaction comparable, so the quench isolates compartment FORMATION
# rather than global collapse.
c_mean = float(models['A'].coupling_matrix().mean())
models['flat'] = Model(np.zeros((N_BEADS,1)), np.array([[0.0]]), c=c_mean)

for k,m in models.items():
    M = m.coupling_matrix()
    print(f'{k:7s} coupling {M.shape}  mean={M.mean():+.4f}  std={M.std():.4f}')

### 5.1 Order parameters

Computed **post hoc** from saved frames — nothing accumulated during the run.

Three parameters, because **no single one detects all four transitions**:

| parameter | definition | detects |
|---|---|---|
| `comp_score` | ½(AA+BB) − AB | *degree* of compartmentalization → **mitotic exit** |
| `comp_asym` | AA − BB | *which* class is compact → **identity swap** |
| `half_asym` | Rg(1st half) − Rg(2nd half) | spatial compaction imbalance → **identity swap** |
| `comp_score` wrt target labels | as above, target masking | contact topology → **rearrangement** |

**Why three (a flaw worth knowing about):** `comp_score` averages AA and BB
symmetrically, so swapping A↔B labels exchanges those two sets and leaves the
value *unchanged*. It is **blind to identity swaps by construction** — verified
offline: config A and its swap both score ≈0.43. `comp_asym` fixes this by
subtracting rather than averaging, exploiting the fact that B–B (≈ −0.33) is
genuinely more attractive than A–A (≈ −0.28); it flips sign under the swap
(−0.31 → +0.34).

The strong test: score against **both** source and target labelling and show a
**crossing** — one falling while the other rises.

In [ ]:
def _pair_masks(N, isA, isB, sep_min=3):
    sep = np.abs(np.subtract.outer(np.arange(N), np.arange(N)))
    ok  = sep >= sep_min
    return (np.outer(isA,isA) & ok, np.outer(isB,isB) & ok,
            (np.outer(isA,isB) | np.outer(isB,isA)) & ok)

def _f(x, mu=MU, rc=RC):
    return 0.5*(1+np.tanh(mu*(rc - squareform(pdist(x)))))

def comp_score(x, masks):
    """Degree of compartmentalization. NOTE: invariant under A<->B identity
    swap (AA and BB are averaged), so it CANNOT detect 'idswap'."""
    f = _f(x); AA, BB, AB = masks
    return 0.5*(f[AA].mean() + f[BB].mean()) - f[AB].mean()

def comp_asym(x, masks):
    """AA - BB. Flips sign under identity swap (B-B more attractive than A-A),
    so this is the discriminating parameter for 'idswap'."""
    f = _f(x); AA, BB, AB = masks
    return f[AA].mean() - f[BB].mean()

def half_asym(x):
    h = len(x)//2
    return radius_of_gyration(x[:h]) - radius_of_gyration(x[h:])

def labels_AB(types):
    t = np.array(types)
    return np.isin(t,['A1','A2']), np.isin(t,['B1','B2','B3','B4'])

masks = {k: _pair_masks(N_BEADS, *labels_AB(v)) for k,v in cfg.items()}

# which parameter is PRIMARY for each scenario
PRIMARY = {'control':'comp_score', 'idswap':'comp_asym',
           'rearrange':'comp_score', 'mitotic':'comp_score'}
print('order parameters defined. primary per scenario:', PRIMARY)

### 5.2 Equilibrate the source ensembles

Two sources are needed: **cell A** (typed) and **flat** (compartment-free).
Full three-phase protocol so each is properly equilibrated before any quench.

`N_REPS` replicas give independent starting configurations — one per replica.
(Drawing many frames from a single replica gives *correlated* starts and inflates
apparent ensemble size without adding information.)

In [ ]:
N_REPS      = 4          # raise for production
N_EQ_STEPS  = 300_000    # source equilibration production length
N_Q_STEPS   = 300_000    # quench observation length
q_spec = SaveSpec(contact_map=True, trajectory=True, velocities=True,
                  forces=False, interval=100, monitor=True)

sim500 = Simulator(seq_file=SEQ500, out_dir=f'{OUT}/transition', platform='cuda')

for src in ['A','flat']:
    for s in range(N_REPS):
        sim500.run_replica(f'src_{src}', s, model=models[src],
                           n_production=N_EQ_STEPS, save_spec=q_spec)
print('source ensembles ready')

### 5.3 Verify the quench mechanism

In [ ]:
report = verify_inplace_quench(SEQ500, models['A'], models['Bswap'],
                               platform='cuda', out_dir=f'{OUT}/_verify')
print(json.dumps(report, indent=2, default=str)[:1500])
print('\nIN-PLACE SUPPORTED' if report['supported'] else
      '\nNOT SUPPORTED -> restart-based quench (velocities re-drawn; discard the'
      '\nfirst frames, which hold a thermal transient, not structural relaxation)')

### 5.4 Run the four quenches

`phases=('production',)` — equilibration MUST NOT run, or the relaxation
transient being measured is destroyed.

In [ ]:
SCENARIOS = {
 'control'  : ('A',    'A',      'null: same H, order params should be stationary'),
 'idswap'   : ('A',    'Bswap',  'identity swap: compaction flips between halves'),
 'rearrange': ('A',    'Cinter', 'contact topology changes: who contacts whom'),
 'mitotic'  : ('flat', 'A',      'de novo compartment formation from flat state'),
}

for tag,(src,dst,desc) in SCENARIOS.items():
    print(f'\n=== {tag}: {desc} ===')
    for s in range(N_REPS):
        start = sim500.load_trajectory(f'src_{src}', s)['xyz'][-1]
        sim500.quench_replica(tag, s, model_B=models[dst], initial_coords=start,
                              n_production=N_Q_STEPS, save_spec=q_spec)

### 5.5 Analysis — order-parameter trajectories

For each scenario, track the compartment score with respect to **both** the
source and target labelling, plus the half-asymmetry. Averaged over replicas.

In [ ]:
SRC_LABELSET = {'control':'A','idswap':'A','rearrange':'A','mitotic':'A'}
DST_LABELSET = {'control':'A','idswap':'Bswap','rearrange':'Cinter','mitotic':'A'}

def traces(tag):
    """Returns dict of arrays (n_reps, n_frames) for every order parameter,
    scored against both source and target labelling."""
    src_m, dst_m = masks[SRC_LABELSET[tag]], masks[DST_LABELSET[tag]]
    out = {k: [] for k in ['score_src','score_dst','asym_src','asym_dst','half']}
    for s in range(N_REPS):
        try: xyz = sim500.load_trajectory(tag, s)['xyz']
        except Exception: continue
        out['score_src'].append([comp_score(x, src_m) for x in xyz])
        out['score_dst'].append([comp_score(x, dst_m) for x in xyz])
        out['asym_src'].append([comp_asym(x, src_m)  for x in xyz])
        out['asym_dst'].append([comp_asym(x, dst_m)  for x in xyz])
        out['half'].append([half_asym(x) for x in xyz])
    return {k: np.array(v) for k,v in out.items()}

results = {t: traces(t) for t in SCENARIOS}
print({t: r['score_src'].shape for t,r in results.items()})

In [ ]:
fig, axes = plt.subplots(3, 4, figsize=(17, 9.5), sharex=True)
for j,(tag,(src,dst,desc)) in enumerate(SCENARIOS.items()):
    R = results[tag]
    if R['score_src'].size == 0: continue
    t = np.arange(R['score_src'].shape[1])
    def band(ax, arr, c, lab):
        m, sd = arr.mean(0), arr.std(0)
        ax.plot(t, m, c=c, label=lab); ax.fill_between(t, m-sd, m+sd, color=c, alpha=.2)

    a = axes[0, j]
    band(a, R['score_src'], 'tab:blue', f"wrt src ({SRC_LABELSET[tag]})")
    if DST_LABELSET[tag] != SRC_LABELSET[tag]:
        band(a, R['score_dst'], 'tab:red', f"wrt tgt ({DST_LABELSET[tag]})")
    star = ' *PRIMARY*' if PRIMARY[tag]=='comp_score' else ''
    a.set_title(f'{tag}{star}\n{desc}', fontsize=8.5); a.legend(fontsize=7)
    if j==0: a.set_ylabel('comp_score\n(degree)')

    b = axes[1, j]
    band(b, R['asym_src'], 'tab:purple', 'asym wrt src')
    b.axhline(0, c='k', lw=.5)
    if PRIMARY[tag]=='comp_asym': b.set_title('*PRIMARY*', fontsize=8, color='crimson')
    if j==0: b.set_ylabel('comp_asym  AA-BB\n(which class compact)')

    c = axes[2, j]
    band(c, R['half'], 'tab:green', 'half asym')
    c.axhline(0, c='k', lw=.5); c.set_xlabel('frame')
    if j==0: c.set_ylabel('Rg(1st)-Rg(2nd)')
plt.tight_layout(); plt.show()
print('idswap: watch comp_asym and half_asym CROSS ZERO (comp_score stays flat —')
print('it is blind to identity swaps by construction).')

### 5.6 Quantify: relaxation times

Fit a stretched exponential to the target-labelling score:

$$y(t) = y_\infty + (y_0 - y_\infty)\,e^{-(t/\tau)^\beta}$$

β < 1 indicates a hierarchy of relaxation timescales — exactly what the 2018
paper found for MiChroM's Rouse-mode correlations (β between 0.4 and 1).

τ is in **frames**; converting to physical time requires the calibration
discussed in the parked dynamics aim (the published water-based calibration runs
1–2 orders of magnitude fast).

In [ ]:
def stretched(t, y0, yinf, tau, beta):
    return yinf + (y0-yinf)*np.exp(-(t/np.maximum(tau,1e-9))**beta)

print(f"{'scenario':11s} {'param':11s} {'y0':>8s} {'yinf':>8s} {'tau':>10s} {'beta':>6s}")
for tag in SCENARIOS:
    R = results[tag]
    if R['score_src'].size == 0: continue
    key = {'comp_score':'score_dst', 'comp_asym':'asym_src'}[PRIMARY[tag]]
    y = R[key].mean(0); t = np.arange(len(y))
    try:
        p,_ = curve_fit(stretched, t, y,
                        p0=[y[0], y[-1], max(len(y)/5,1), 0.7], maxfev=20000)
        print(f'{tag:11s} {PRIMARY[tag]:11s} {p[0]:8.4f} {p[1]:8.4f} {p[2]:10.1f} {p[3]:6.2f}')
    except Exception as e:
        print(f'{tag:11s} {PRIMARY[tag]:11s} fit failed: {e}')
print('\ntau in FRAMES. Control tau should be ill-defined/huge (no relaxation) —')
print('that is the null working. beta<1 = hierarchy of timescales (2018 paper: 0.4-1).')

### 5.7 Before / after contact maps

Visual confirmation that the ensemble actually moved. Early vs late frames of the
quench, O/E, ordered by the **target** labelling.

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 7.5))
for j,tag in enumerate(SCENARIOS):
    try: xyz = sim500.load_trajectory(tag, 0)['xyz']
    except Exception: continue
    n = len(xyz); early = contact_probability(xyz[:n//5]); late = contact_probability(xyz[-n//5:])
    tl = np.array(cfg[DST_LABELSET[tag]])
    order = np.concatenate([np.where(np.isin(tl,['A1','A2']))[0],
                            np.where(np.isin(tl,['B1','B2']))[0]])
    for i,(M,lab) in enumerate([(early,'early'),(late,'late')]):
        OE = observed_over_expected(M)[np.ix_(order,order)]
        with np.errstate(divide='ignore', invalid='ignore'):
            axes[i,j].imshow(np.log2(OE), cmap='RdBu_r', vmin=-.8, vmax=.8)
        axes[i,j].set_title(f'{tag} — {lab}', fontsize=9)
        axes[i,j].axhline((np.isin(tl,['A1','A2'])).sum(), c='k', lw=.5)
        axes[i,j].axvline((np.isin(tl,['A1','A2'])).sum(), c='k', lw=.5)
plt.tight_layout(); plt.show()
print('Ordered by TARGET labels: a completed transition shows a clean two-block'
      '\ncheckerboard in the LATE row and not in the EARLY row.')

## 6. Summary

One `Simulator`, one physics implementation, validated once:

- **statics** — `SAVE_MAPS_ONLY`, three phases → reproduces k2
- **dynamics** — float32 positions + velocities, uniform interval
- **transitions** — four quench scenarios differing only in **C**
- **any potential** — swap the `Model`

Every run writes a metadata sidecar (integrator, γ, temperature, mass, force list,
versions, seed, convergence), so runs are reproducible and forces recomputable
even when not saved.

**Design notes worth keeping**
- The **control quench is the null** that makes the other scenarios interpretable.
- Scoring against **both** source and target labellings, and showing a crossing,
  is a far stronger claim than a single order parameter moving.
- The **flat → typed** (mitotic-exit) scenario uses a constant coupling equal to
  the mean, so overall compaction is preserved and the quench isolates compartment
  *formation* rather than global collapse.
- 100% type switching is an extreme caricature (real differentiation flips ~10–20%
  of loci); it is chosen here for visibility while validating the machinery. The
  fraction-switched is the natural knob for the planned composition/time sweep.

**Next:** kernel optimization (fit Λ for fixed C), now that the forward path covers
every experiment type.